# US Census Block Dataset Builder

Builds a block-level urban-analysis dataset (Census 2020 PL + DHC, ACS 2023 5-year, OpenStreetMap, Microsoft Building Footprints, GHSL) for one US county or one census tract.

1. Get a **configuration token** from the geography picker: <https://omid-mansourihanis.github.io/us-census-block-builder/>
2. Paste the token and your free **Census API key** into Cell 1.
3. **Runtime → Run all.** The GeoPackage + CSV download automatically at the end.

Source code: [github.com/Omid-Mansourihanis/Omid-Mansourihanis.github.io/tree/main/us-census-block-builder](https://github.com/Omid-Mansourihanis/Omid-Mansourihanis.github.io/tree/main/us-census-block-builder)

In [ ]:
# %% CELL 1 — paste your token + Census key, then Run All
# --------------------------------------------------------------------------
# Paste the token you copied from the geography picker web page:
CONFIG_TOKEN = "PASTE_YOUR_TOKEN_HERE"

# Paste your free Census API key (https://api.census.gov/data/key_signup.html):
CENSUS_API_KEY = "PASTE_YOUR_CENSUS_KEY_HERE"

# Which built-environment layers to include (turn off the heavy ones for a quick test):
INCLUDE_OSM       = True    # OpenStreetMap streets + POIs (~minutes)
INCLUDE_BUILDINGS = True    # Microsoft Building Footprints (per-state, hundreds of MB)
INCLUDE_GHSL      = True    # GHSL height/volume/urbanization (~2 GB first time)

# --- decode the token (no edits needed below) ---
import json, base64
def _decode(tok):
    pad = "=" * (-len(tok) % 4)
    raw = base64.urlsafe_b64decode(tok + pad)
    return json.loads(raw)

assert CONFIG_TOKEN != "PASTE_YOUR_TOKEN_HERE", "Paste your configuration token first."
assert CENSUS_API_KEY and CENSUS_API_KEY != "PASTE_YOUR_CENSUS_KEY_HERE", "Paste your Census API key first."
CFG = _decode(CONFIG_TOKEN)
print("✓ Configuration decoded:")
for k, v in CFG.items():
    print(f"    {k:12}: {v}")
if CFG.get("tract_fips"):
    print("\n  Mode: SINGLE TRACT (fast test build)")
else:
    print("\n  Mode: WHOLE COUNTY (full build — several GB, 10–30 min)")

In [ ]:
# %% CELL 2 — install dependencies + fetch the pipeline from your repo
# --------------------------------------------------------------------------
# Raw URL of the pipeline module (already set; change only if you fork the repo)
PIPELINE_RAW_URL = (
    "https://raw.githubusercontent.com/Omid-Mansourihanis/Omid-Mansourihanis.github.io/"
    "main/us-census-block-builder/census_pipeline_modular.py"
)

import subprocess, sys
print("→ Installing geospatial dependencies (1–2 min)…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pygris", "geopandas", "osmnx", "rasterio",
                "requests", "pandas", "shapely", "tqdm"], check=True)

import urllib.request, os
print("→ Fetching pipeline module from repo…")
try:
    urllib.request.urlretrieve(PIPELINE_RAW_URL, "census_pipeline_modular.py")
    assert os.path.getsize("census_pipeline_modular.py") > 1000
    print("  ✓ census_pipeline_modular.py downloaded")
except Exception as e:
    print(f"  ✗ Could not fetch the pipeline module: {e}")
    print("    Fix PIPELINE_RAW_URL in this cell, or upload the .py manually via the Files panel.")
    raise

In [ ]:
# %% CELL 3 — run the pipeline
# --------------------------------------------------------------------------
from census_pipeline_modular import run_county

state  = CFG["state_fips"]
county = CFG["county_fips"]
tract  = CFG.get("tract_fips")
cname  = CFG.get("county_name") or f"{state}{county}"

if not tract:
    # ---- WHOLE COUNTY: the normal, supported path ----
    gdf = run_county(
        state, county,
        census_api_key   = CENSUS_API_KEY,
        county_name      = cname,
        include_osm      = INCLUDE_OSM,
        include_buildings= INCLUDE_BUILDINGS,
        include_ghsl     = INCLUDE_GHSL,
    )
else:
    # ---- SINGLE TRACT: lightweight Census-only build for quick testing ----
    # run_county operates at county scope, so for a single tract we pull the
    # core Census block attributes directly and filter to the chosen tract.
    # (Built-environment layers are county-scoped and skipped in tract mode;
    #  run a whole-county build to include OSM / buildings / GHSL.)
    print("Single-tract mode: building Census-only block table for tract", tract)
    import census_pipeline_modular as cpm
    import pandas as pd, pygris
    api = cpm.CensusAPI(CENSUS_API_KEY)
    AL  = cpm.build_alias_maps()

    pl = api.fetch("2020/dec/pl", list(AL["PL"]), "block", state, county,
                   tract_fips=tract)          # ask the API for this tract only
    pl["GEOID20"] = pl["state"]+pl["county"]+pl["tract"]+pl["block"]
    print(f"  {len(pl):,} blocks in tract {tract}")

    blocks = pygris.blocks(state=state, county=county, year=2020)
    blocks["GEOID20"] = blocks["GEOID20"].astype(str)
    blocks = blocks[blocks["TRACTCE20"].astype(str) == tract].copy()
    gdf = blocks.merge(
        pl.drop(columns=["state","county","tract","block","NAME"], errors="ignore"),
        on="GEOID20", how="left")
    gdf = gdf.rename(columns={**AL["PL"],
        "STATEFP20":"State_FIPS","COUNTYFP20":"County_FIPS",
        "TRACTCE20":"Tract_Code","BLOCKCE20":"Block_Code",
        "GEOID20":"GeoID","NAME20":"Block_Name"})
    out_dir = "./census_pipeline_output"; os.makedirs(out_dir, exist_ok=True)
    stem = f"census_{cname}_{state}{county}_tract{tract}"
    gdf.to_file(f"{out_dir}/{stem}.gpkg", driver="GPKG")
    pd.DataFrame(gdf.drop(columns="geometry")).to_csv(f"{out_dir}/{stem}.csv", index=False)
    print(f"  ✓ wrote {out_dir}/{stem}.gpkg")

print(f"\n✓ DONE — {len(gdf):,} blocks × {len(gdf.columns)} columns")

In [ ]:
# %% CELL 4 — download the result
# --------------------------------------------------------------------------
import glob, os
produced = sorted(glob.glob("./census_pipeline_output/*.gpkg") +
                  glob.glob("./census_pipeline_output/*.csv"))
print("Files produced:")
for f in produced:
    print(f"  {f}  ({os.path.getsize(f)/1e6:.1f} MB)")

# Download the dataset (GeoPackage + CSV + column metadata), not the cached
# building-footprint extract.
try:
    from google.colab import files
except ImportError:
    files = None
    print("Not running in Colab: the files are in ./census_pipeline_output/")
for f in produced:
    if files and "_buildings" not in os.path.basename(f):
        try:
            files.download(f)
        except Exception as e:
            print(f"  (could not auto-download {f}: {e} — use the Files panel on the left)")